# 04 — Aggregation & Export

Aggregates gold listing-level tables to LAD and MSOA level for dashboard and app consumption. Also computes STR vs LTR yield estimates per area. Exports final tables to the `airbnb_app.export` schema.

**Catalog:** `airbnb_app`  
**Reads from:** `airbnb_app.gold.airbnb_listings_{city}`  
**Writes to:** `airbnb_app.export.*`  

| Step | What happens |
|------|-------------|
| A | Aggregate listing-level gold tables to MSOA and LAD granularity |
| B | Compute STR yield and LTR yield estimates per area |
| C | Edinburgh supplementary enrichment (house price / rent from manual file) |
| D | Write all export tables to `airbnb_app.export` |
| E | Spot checks and summary |

> ⚠️ **Photon UNION ALL bug:** per-city loops used throughout — do not replace with cross-city UNION ALL `.show()` calls.
> **Edinburgh:** `msoa_code` is null throughout. MSOA-level aggregations exclude Edinburgh. LAD-level aggregations include Edinburgh.

## 0. Config

In [0]:
GOLD_DB   = "airbnb_app.gold"
EXPORT_DB = "airbnb_app.export"
rent_pd = spark.table("airbnb_app.clean.rent_data").toPandas()
CITIES = ["london", "manchester", "edinburgh", "bristol"]

# Edinburgh supplementary file — house price and rent data (manual upload)
# Scotland uses Data Zones, not MSOAs — this file provides LAD-level estimates
EDINBURGH_ENRICHMENT_PATH = "/Volumes/airbnb_app/raw/edinburgh/edinburgh_house_price_rent.csv"

# STR yield assumptions
# Occupancy rate applied to active listings to estimate annual STR revenue
ASSUMED_OCCUPANCY_RATE = 0.65   # 65% — conservative estimate for UK STR market
NIGHTS_PER_YEAR        = 365

# LTR yield: median annual rent / median house price
# Rent estimates sourced from ONS private rental market statistics (LAD level)
# Loaded from clean layer in Part B

## 1. Setup

In [0]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {EXPORT_DB}")
print(f"Schema ready: {EXPORT_DB}")

export_log = []

---
## Part A — Aggregate to MSOA and LAD Level

### A1. MSOA-level aggregation (England and Wales: London, Manchester, Bristol)

In [0]:
msoa_frames = []

EW_CITIES = [c for c in CITIES if c != "edinburgh"]

for city in EW_CITIES:
    src = f"{GOLD_DB}.airbnb_listings_{city}"
    df = spark.table(src).filter(F.col("msoa_code").isNotNull())

    agg = df.groupBy("msoa_code", "msoa_name", "lad_code", "lad_name").agg(
        F.count("id").alias("total_listings"),
        F.round(F.avg("price"), 2).alias("avg_nightly_price"),
        F.round(F.median("price"), 2).alias("median_nightly_price"),
        F.round(F.avg("review_scores_rating"), 2).alias("avg_review_score"),
        F.round(F.avg("availability_365"), 1).alias("avg_availability_365"),
        F.first("median_house_price_2025").alias("median_house_price_2025"),
        F.first("median_house_price_2015").alias("median_house_price_2015"),
        F.first("price_growth_10yr").alias("price_growth_10yr"),
        F.first("less_than_15_minute_walk").alias("less_than_15_minute_walk"),
        F.first("less_than_30_minute_walk").alias("less_than_30_minute_walk"),
        F.first("gp_surgery_count").alias("gp_surgery_count"),
        F.first("total_parks_count").alias("total_parks_count"),
    ).withColumn("city", F.lit(city))

    msoa_frames.append(agg)
    print(f"  {city}: {agg.count():,} MSOAs")

msoa_combined = msoa_frames[0]
for frame in msoa_frames[1:]:
    msoa_combined = msoa_combined.unionByName(frame)

print(f"\nTotal MSOA rows: {msoa_combined.count():,}")

### A2. LAD-level aggregation (all four cities including Edinburgh)

In [0]:
lad_frames = []

for city in CITIES:
    src = f"{GOLD_DB}.airbnb_listings_{city}"
    df = spark.table(src).filter(F.col("lad_code").isNotNull())

    agg = df.groupBy("lad_code", "lad_name").agg(
        F.count("id").alias("total_listings"),
        F.round(F.avg("price"), 2).alias("avg_nightly_price"),
        F.round(F.median("price"), 2).alias("median_nightly_price"),
        F.round(F.avg("review_scores_rating"), 2).alias("avg_review_score"),
        F.round(F.avg("availability_365"), 1).alias("avg_availability_365"),
        F.first("gp_surgery_count").alias("gp_surgery_count"),
        F.first("gps_per_100000_people").alias("gps_per_100000_people"),
        F.first("total_parks_count").alias("total_parks_count"),
        F.first("parks_and_play_areas_per_100000_people").alias("parks_per_100000_people"),
    ).withColumn("city", F.lit(city))

    lad_frames.append(agg)
    print(f"  {city}: {agg.count():,} LADs")

lad_combined = lad_frames[0]
for frame in lad_frames[1:]:
    lad_combined = lad_combined.unionByName(frame)

print(f"\nTotal LAD rows: {lad_combined.count():,}")

---
## Part B — STR vs LTR Yield Estimates

### B1. Load ONS rental data (LAD level)

In [0]:
# ONS private rental market statistics — median monthly rent by LAD
# Loaded from clean layer
rent_pd = spark.table("airbnb_app.clean.rent_data").toPandas()

# Filter to most recent period and rename columns to match pipeline conventions
rent_pd = (
    rent_pd.sort_values("period_date", ascending=False)
           .drop_duplicates(subset="area_code")
           .rename(columns={"area_code": "lad_code", "rental_price": "median_monthly_rent"})
[["lad_code", "median_monthly_rent"]]
)

print(f"  Rent data (LAD): {rent_pd.shape}")
print(rent_pd.head(3))
print(f"  ONS rents (LAD): {rent_pd.shape}")
print(rent_pd.head(3))

### B2. Compute STR and LTR yields at MSOA level

In [0]:
# STR annual revenue estimate:
#   median_nightly_price * occupancy_rate * 365
# STR gross yield:
#   str_annual_revenue / median_house_price_2025
#
# LTR gross yield:
#   (median_monthly_rent * 12) / median_house_price_2025

msoa_pd = msoa_combined.toPandas()

# Merge LAD-level rent onto MSOA table (rent is LAD-level — same value repeats per LAD)
msoa_pd = msoa_pd.merge(
    rent_pd[["lad_code", "median_monthly_rent"]],
    on="lad_code", how="left"
)

# STR yield
msoa_pd["str_annual_revenue_est"] = (
    msoa_pd["median_nightly_price"] * ASSUMED_OCCUPANCY_RATE * NIGHTS_PER_YEAR
).round(0)

msoa_pd["str_gross_yield"] = (
    msoa_pd["str_annual_revenue_est"] / msoa_pd["median_house_price_2025"]
).round(4)

# LTR yield
msoa_pd["ltr_annual_revenue_est"] = (msoa_pd["median_monthly_rent"] * 12).round(0)

msoa_pd["ltr_gross_yield"] = (
    msoa_pd["ltr_annual_revenue_est"] / msoa_pd["median_house_price_2025"]
).round(4)

# Yield advantage: STR over LTR
msoa_pd["str_vs_ltr_yield_delta"] = (
    msoa_pd["str_gross_yield"] - msoa_pd["ltr_gross_yield"]
).round(4)

print(f"MSOA yield table: {msoa_pd.shape}")
print(msoa_pd[["msoa_name", "city", "median_nightly_price", "str_gross_yield",
               "ltr_gross_yield", "str_vs_ltr_yield_delta"]].head(5))

### B3. Compute STR and LTR yields at LAD level

In [0]:
lad_pd = lad_combined.toPandas()

# For LAD-level yield we need house prices — aggregate from MSOA medians
# (Edinburgh: no MSOA house prices — patched in Part C)
msoa_hp = msoa_pd.groupby("lad_code")["median_house_price_2025"].median().reset_index()
msoa_hp.columns = ["lad_code", "median_house_price_2025_lad"]

lad_pd = lad_pd.merge(msoa_hp, on="lad_code", how="left")
lad_pd = lad_pd.merge(rent_pd[["lad_code", "median_monthly_rent"]], on="lad_code", how="left")

lad_pd["str_annual_revenue_est"] = (
    lad_pd["median_nightly_price"] * ASSUMED_OCCUPANCY_RATE * NIGHTS_PER_YEAR
).round(0)

lad_pd["str_gross_yield"] = (
    lad_pd["str_annual_revenue_est"] / lad_pd["median_house_price_2025_lad"]
).round(4)

lad_pd["ltr_annual_revenue_est"] = (lad_pd["median_monthly_rent"] * 12).round(0)

lad_pd["ltr_gross_yield"] = (
    lad_pd["ltr_annual_revenue_est"] / lad_pd["median_house_price_2025_lad"]
).round(4)

lad_pd["str_vs_ltr_yield_delta"] = (
    lad_pd["str_gross_yield"] - lad_pd["ltr_gross_yield"]
).round(4)

print(f"LAD yield table: {lad_pd.shape}")

---
## Part C — Edinburgh Supplementary Enrichment

In [0]:
# Edinburgh has no MSOA house prices or ONS rail data (Scotland uses Data Zones).
# A manually curated file provides LAD-level house price and rent estimates
# sourced from Registers of Scotland and Scottish Government rental statistics.

import os

if os.path.exists(EDINBURGH_ENRICHMENT_PATH):
    edin_pd = pd.read_csv(EDINBURGH_ENRICHMENT_PATH)
    print(f"  Edinburgh enrichment loaded: {edin_pd.shape}")
    print(edin_pd)

    # Patch LAD yield table for Edinburgh rows
    edin_lad_code = lad_pd.loc[lad_pd["city"] == "edinburgh", "lad_code"].iloc[0]
    edin_hp   = edin_pd["median_house_price_2025"].iloc[0]
    edin_rent = edin_pd["median_monthly_rent"].iloc[0]

    mask = lad_pd["city"] == "edinburgh"
    lad_pd.loc[mask, "median_house_price_2025_lad"] = edin_hp
    lad_pd.loc[mask, "median_monthly_rent"]          = edin_rent

    # Recompute yields for Edinburgh rows
    lad_pd.loc[mask, "str_annual_revenue_est"] = (
        lad_pd.loc[mask, "median_nightly_price"] * ASSUMED_OCCUPANCY_RATE * NIGHTS_PER_YEAR
    ).round(0)
    lad_pd.loc[mask, "ltr_annual_revenue_est"] = (edin_rent * 12)
    lad_pd.loc[mask, "str_gross_yield"] = (
        lad_pd.loc[mask, "str_annual_revenue_est"] / edin_hp
    ).round(4)
    lad_pd.loc[mask, "ltr_gross_yield"] = (
        lad_pd.loc[mask, "ltr_annual_revenue_est"] / edin_hp
    ).round(4)
    lad_pd.loc[mask, "str_vs_ltr_yield_delta"] = (
        lad_pd.loc[mask, "str_gross_yield"] - lad_pd.loc[mask, "ltr_gross_yield"]
    ).round(4)

    print(f"  Edinburgh yields patched.")

else:
    print(f"  WARNING: Edinburgh enrichment file not found at {EDINBURGH_ENRICHMENT_PATH}")
    print(f"  Edinburgh yield columns will be null in the LAD export table.")

---
## Part D — Write Export Tables

In [0]:
def write_export(pdf: pd.DataFrame, table_name: str, description: str):
    """Cast float32 → float64, create Spark DataFrame, write to export schema."""
    for c in pdf.select_dtypes(include="float32").columns:
        pdf[c] = pdf[c].astype("float64")

    sdf = spark.createDataFrame(pdf)
    sdf = sdf.withColumn("_export_created_at", F.current_timestamp())

    tgt = f"{EXPORT_DB}.{table_name}"
    (
        sdf.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tgt)
    )
    rows = sdf.count()
    print(f"  ✓ {tgt} — {rows:,} rows  ({description})")
    export_log.append({"table": tgt, "rows": rows, "description": description})


print("Writing export tables...")
write_export(msoa_pd,  "msoa_investment_summary",  "MSOA-level aggregation + STR/LTR yields")
write_export(lad_pd,   "lad_investment_summary",   "LAD-level aggregation + STR/LTR yields (all cities)")
print("\nExport writes complete.")

---
## Part E — Spot Checks & Summary

In [0]:
# Top 10 MSOAs by STR gross yield
spark.sql("""
    SELECT city, msoa_name, lad_name,
           median_nightly_price, str_gross_yield, ltr_gross_yield, str_vs_ltr_yield_delta
    FROM airbnb_app.export.msoa_investment_summary
    WHERE str_gross_yield IS NOT NULL
    ORDER BY str_gross_yield DESC
    LIMIT 10
""").display()

In [0]:
# LAD summary — all four cities, key yield columns
spark.sql("""
    SELECT city, lad_name, total_listings, avg_nightly_price,
           str_gross_yield, ltr_gross_yield, str_vs_ltr_yield_delta,
           gp_surgery_count, total_parks_count
    FROM airbnb_app.export.lad_investment_summary
    ORDER BY city, str_gross_yield DESC
""").display()

In [0]:
# Null audit on LAD export — Edinburgh yield columns should now be populated
for city in CITIES:
    print(f"\n{city.upper()}")
    spark.sql(f"""
        SELECT
            COUNT(*)                                                         AS total_lads,
            SUM(CASE WHEN str_gross_yield IS NULL THEN 1 ELSE 0 END)         AS missing_str_yield,
            SUM(CASE WHEN ltr_gross_yield IS NULL THEN 1 ELSE 0 END)         AS missing_ltr_yield,
            SUM(CASE WHEN median_house_price_2025_lad IS NULL THEN 1 ELSE 0 END) AS missing_house_price
        FROM airbnb_app.export.lad_investment_summary
        WHERE city = '{city}'
    """).display()

In [0]:
# Export summary log
summary = pd.DataFrame(export_log)
display(summary)

print("\nAggregation and export complete.")
print("\nNext step: build Streamlit app consuming airbnb_app.export tables.")

In [0]:
# Export to CSV
EXPORT_CSV_PATH = "/Volumes/airbnb_app/gold/exports/"

msoa_pd.to_csv(f"{EXPORT_CSV_PATH}msoa_investment_summary.csv", index=False)
lad_pd.to_csv(f"{EXPORT_CSV_PATH}lad_investment_summary.csv", index=False)

print(f"CSVs written to {EXPORT_CSV_PATH}")

# Downloadable links
displayHTML(f"""
    <a href='/files/Volumes/airbnb_app/gold/exports/msoa_investment_summary.csv'>
        Download MSOA Investment Summary CSV</a><br>
    <a href='/files/Volumes/airbnb_app/gold/exports/lad_investment_summary.csv'>
        Download LAD Investment Summary CSV</a>
""")

In [0]:
import base64

def make_download_link(csv_path: str, filename: str) -> str:
    with open(csv_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    return f'<a href="data:text/csv;base64,{b64}" download="{filename}">{filename}</a>'

VOLUME_PATH = "/Volumes/airbnb_app/gold/exports/"

msoa_pd.to_csv(f"{VOLUME_PATH}msoa_investment_summary.csv", index=False)
lad_pd.to_csv(f"{VOLUME_PATH}lad_investment_summary.csv", index=False)

displayHTML(
    make_download_link(f"{VOLUME_PATH}msoa_investment_summary.csv", "msoa_investment_summary.csv")
    + "<br><br>" +
    make_download_link(f"{VOLUME_PATH}lad_investment_summary.csv", "lad_investment_summary.csv")
)

## Notes

- **MSOA aggregation** covers London, Manchester, Bristol only — Edinburgh excluded (no MSOA codes).
- **LAD aggregation** covers all four cities. Edinburgh house price and rent patched from manual CSV in Part C.
- **STR yield** = `(median_nightly_price × 0.65 × 365) / median_house_price_2025`. Occupancy rate (65%) is a conservative UK STR market assumption — adjust `ASSUMED_OCCUPANCY_RATE` in Config if needed.
- **LTR yield** = `(median_monthly_rent × 12) / median_house_price_2025`. Rent figures sourced from ONS private rental market statistics at LAD level.
- **`str_vs_ltr_yield_delta`** is the key investment signal: positive = STR outperforms LTR on gross yield.
- **House prices at LAD level** are derived by taking the median of MSOA-level medians within each LAD — appropriate given MSOA data is the native granularity of the Land Registry price paid dataset.
- **Photon UNION ALL bug:** spot checks and null audits loop per-city throughout this notebook. Do not replace with cross-city UNION ALL `.show()` calls.
- **Next step:** Streamlit app consuming `airbnb_app.export.msoa_investment_summary` and `airbnb_app.export.lad_investment_summary`.